In [1]:
"""Run the uploaded LSTM-ACI experiment on ONE dataset and audit Eq. (23).

Requires the same Helper_Scripts/models_TLMR.py as the original experiment.
Change DATASET_PATH then: python validate_aci_eq23.py
"""
from pathlib import Path
from bisect import bisect_right, insort
import random
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

from Helper_Scripts.models_TLMR import LSTMQuantileReg

# --------------------------- CONFIG ---------------------------
DATASET_PATH = '../Datasets/50Hertz.csv'  # <-- SET YOUR CSV PATH OR PUBLIC CSV URL
OUTPUT_DIR = Path('./aci_eq23_validation')
SEQ_LEN = 48
EPOCHS = 200
LR = 0.001
SEED = 42
BATCH_SIZE = 32
ALPHA = 0.10
TINIT = 5000
SPLIT = 0.5
GAMMA = 0.001
DOWNSAMPLE_STRIDE = 4  # matches uploaded experiment
WINDOW = 24
M_DECAY = 0.95  # Eq. (22), NOT the LACI code's 0.85
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# --------------------------------------------------------------


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class SeqDataset(Dataset):
    def __init__(self, series, seq_len):
        self.X = np.array([series[i:i + seq_len] for i in range(len(series) - seq_len)], dtype=np.float32)
        self.y = np.array([series[i + seq_len] for i in range(len(series) - seq_len)], dtype=np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.y[idx])


def load_dataset(path):
    # Exact preprocessing used in the uploaded script: discard column 0,
    # flatten the remaining columns row-wise, then keep every fourth value.
    df = pd.read_csv(path)
    if df.shape[1] < 2:
        raise ValueError('Expected a CSV with a timestamp/index column followed by numeric data columns.')
    data = df.iloc[:, 1:].apply(pd.to_numeric, errors='raise').to_numpy().flatten().astype(float)
    if not np.isfinite(data).all():
        raise ValueError('CSV contains missing/non-finite values; clean it before running ACI.')
    data = data[::DOWNSAMPLE_STRIDE]
    if len(data) <= TINIT + SEQ_LEN:
        raise ValueError(f'Too few values after downsampling: {len(data)}; need > {TINIT + SEQ_LEN}')
    scaler = StandardScaler()
    scaler.fit(data[:TINIT + SEQ_LEN].reshape(-1, 1))
    scaled = scaler.transform(data.reshape(-1, 1)).flatten()
    return SeqDataset(scaled, SEQ_LEN), scaler


def quantile_loss(y, pred, q):
    d = y - pred
    return torch.mean(torch.maximum(q * d, (q - 1) * d))


def fresh_models():
    lower = LSTMQuantileReg().to(DEVICE)
    upper = LSTMQuantileReg().to(DEVICE)
    return lower, upper, torch.optim.Adam(lower.parameters(), lr=LR), torch.optim.Adam(upper.parameters(), lr=LR)


def maybe_train(t, x_train, y_train, lower, upper, opt_lo, opt_hi):
    # Matches original: 200 epochs once; one additional epoch every 6 steps.
    epochs = EPOCHS if t == TINIT else (1 if (t - TINIT) % 6 == 0 else 0)
    if epochs == 0:
        return
    x = torch.as_tensor(x_train.reshape(-1, SEQ_LEN), dtype=torch.float32, device=DEVICE)
    y = torch.as_tensor(y_train.reshape(-1, 1), dtype=torch.float32, device=DEVICE)
    loader = DataLoader(TensorDataset(x, y), batch_size=min(BATCH_SIZE, len(y)), shuffle=True)
    for model, opt, q in [(lower, opt_lo, ALPHA / 2), (upper, opt_hi, 1 - ALPHA / 2)]:
        model.train()
        for _ in range(epochs):
            for xb, yb in loader:
                loss = quantile_loss(yb, model(xb), q)
                opt.zero_grad()
                loss.backward()
                opt.step()


def run_aci(ds, scaler):
    """Reproduces run_aci() from the uploaded script (unweighted standard ACI)."""
    set_seed()
    lower, upper, opt_lo, opt_hi = fresh_models()
    current_alpha = ALPHA
    rows = []
    start = time.time()
    for t in range(TINIT, len(ds)):
        idx = np.arange(t - TINIT, t)
        n_train = int(round(SPLIT * TINIT))
        train_idx, cal_idx = idx[:n_train], idx[n_train:]
        maybe_train(t, ds.X[train_idx], ds.y[train_idx], lower, upper, opt_lo, opt_hi)
        with torch.no_grad():
            lower.eval()
            upper.eval()
            xc = torch.as_tensor(ds.X[cal_idx].reshape(-1, SEQ_LEN), dtype=torch.float32, device=DEVICE)
            qlc = lower(xc).cpu().numpy().reshape(-1)
            quc = upper(xc).cpu().numpy().reshape(-1)
            xt = torch.as_tensor(ds.X[t].reshape(1, SEQ_LEN), dtype=torch.float32, device=DEVICE)
            qlo = float(lower(xt).cpu().numpy()[0, 0])
            qhi = float(upper(xt).cpu().numpy()[0, 0])
        scores = np.maximum(qlc - ds.y[cal_idx], ds.y[cal_idx] - quc)
        alpha_issued = float(np.clip(current_alpha, 1e-6, 1 - 1e-6))
        level = float(np.clip((1 - alpha_issued) * (1 + 1 / len(cal_idx)), 0, 1))
        radius = float(np.quantile(scores, level))
        pred_lo, pred_hi = qlo - radius, qhi + radius
        y = float(ds.y[t])
        err = int(not (pred_lo <= y <= pred_hi))
        current_alpha = float(np.clip(alpha_issued + GAMMA * (ALPHA - err), 1e-6, 1 - 1e-6))
        rows.append((t - TINIT + 1, t, y, pred_lo, pred_hi, err, alpha_issued, current_alpha))
        if (t - TINIT + 1) % 200 == 0:
            print(f'ACI predictions: {t-TINIT+1}/{len(ds)-TINIT} | alpha={current_alpha:.4f}', flush=True)
    traj = pd.DataFrame(rows, columns=['i', 'dataset_index', 'y_true_scaled', 'y_l_scaled', 'y_u_scaled', 'err', 'alpha_issued', 'alpha_after_update'])
    traj['covered'] = 1 - traj['err']
    offset, scale = float(scaler.mean_[0]), float(scaler.scale_[0])
    for name, outname in [('y_true_scaled', 'y_true'), ('y_l_scaled', 'y_l_pred'), ('y_u_scaled', 'y_u_pred')]:
        traj[outname] = traj[name] * scale + offset
    print(f'ACI finished in {time.time()-start:.1f}s; global coverage={traj.covered.mean():.4f}')
    return traj


def validate_eq23(traj):
    """At each update i, compare ALL historical failure LE_s to ONE LEbar_i.

    For the first WINDOW-1 points, LE_s is set to 0, matching the
    warm-up convention in the uploaded LACI implementation. Report an
    additional eligible-only audit that excludes those early indices.
    """
    covered = traj['covered'].to_numpy(dtype=np.float64)
    errors = traj['err'].to_numpy(dtype=int)
    n = len(traj)
    local_error = np.zeros(n, dtype=float)
    if n >= WINDOW:
        counts = np.convolve(covered, np.ones(WINDOW), mode='valid')
        local_error[WINDOW-1:] = np.maximum((1 - ALPHA) - counts / WINDOW, 0.0)

    sorted_failure_le = []
    sorted_eligible_failure_le = []
    weighted_numerator = 0.0
    weighted_denominator = 0.0
    out = []
    for j in range(n):
        le = float(local_error[j])
        weighted_numerator = M_DECAY * weighted_numerator + le
        weighted_denominator = M_DECAY * weighted_denominator + 1.0
        lebar = weighted_numerator / weighted_denominator

        if errors[j]:
            insort(sorted_failure_le, le)
            if j >= WINDOW - 1:
                insort(sorted_eligible_failure_le, le)
        n_fail = len(sorted_failure_le)
        hits = n_fail - bisect_right(sorted_failure_le, lebar)  # strict >
        prop = hits / n_fail if n_fail else np.nan
        p_min = n_fail / hits if hits else np.inf
        n_eligible = len(sorted_eligible_failure_le)
        hits_eligible = n_eligible - bisect_right(sorted_eligible_failure_le, lebar)
        prop_eligible = hits_eligible / n_eligible if n_eligible else np.nan

        out.append((j+1, le, lebar, int(errors[j]), n_fail, hits,
                    prop, p_min, bool(2*hits > n_fail) if n_fail else False,
                    n_eligible, hits_eligible, prop_eligible,
                    bool(2*hits_eligible > n_eligible) if n_eligible else False))

    audit = pd.DataFrame(out, columns=[
        'i', 'LE_i', 'LEbar_i', 'err_i', 'n_failures_F_i', 'count_LEs_gt_LEbar_i',
        'fraction', 'p_min', 'exists_p_lt_2', 'eligible_failures',
        'eligible_count_gt', 'eligible_fraction', 'eligible_exists_p_lt_2'])
    # All comparisons use the left-sided full 24-observation window; do not
    # call the first WINDOW-1 updates valid/invalid when LE is not defined.
    valid = audit.iloc[WINDOW-1:].copy()
    final = valid.iloc[-1]
    failures = audit.loc[audit.err_i.eq(1) & audit.i.ge(WINDOW)]
    summary = {
        'n_predictions': n,
        'n_updates_evaluated': len(valid),
        'global_coverage': float(covered.mean()),
        'total_failures_all': int(errors.sum()),
        'final_F_i_size': int(final.n_failures_F_i),
        'final_count_condition_true': int(final.count_LEs_gt_LEbar_i),
        'final_fraction': float(final.fraction),
        'final_p_min': float(final.p_min),
        'final_supports_p_lt_2': bool(final.exists_p_lt_2),
        'fraction_updates_supporting_p_lt_2': float(valid.exists_p_lt_2.mean()),
        'final_eligible_failure_count': int(final.eligible_failures),
        'final_eligible_hits': int(final.eligible_count_gt),
        'final_eligible_fraction': float(final.eligible_fraction),
        'final_eligible_supports_p_lt_2': bool(final.eligible_exists_p_lt_2),
        'failure_time_fraction_LEi_gt_LEbar_i': float((failures.LE_i > failures.LEbar_i).mean()) if len(failures) else np.nan,
        'failure_time_mean_LEi_minus_LEbar_i': float((failures.LE_i - failures.LEbar_i).mean()) if len(failures) else np.nan,
    }
    return audit, summary


def make_plots(audit, outdir):
    valid = audit.iloc[WINDOW-1:]
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(valid.i, valid.fraction, label='Historical failure fraction satisfying Eq. (23)', linewidth=1.5)
    ax.axhline(0.5, linestyle='--', label='Threshold: p = 2')
    ax.set(xlabel='Online update i', ylabel='Fraction', ylim=(-0.02, 1.02), title='ACI: historical failure majority against fixed LEbar_i')
    ax.legend()
    fig.tight_layout()
    fig.savefig(outdir / 'eq23_fraction_over_time.png', dpi=180)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(valid.i, valid.LE_i, label='Local error LE_i', linewidth=1.3)
    ax.plot(valid.i, valid.LEbar_i, label='Recency-weighted LEbar_i', linewidth=1.3)
    bad = valid[valid.err_i.eq(1) & valid.LE_i.gt(valid.LEbar_i)]
    ax.scatter(bad.i, bad.LE_i, s=9, label='Current ACI failure with LE_i > LEbar_i')
    ax.set(xlabel='Online update i', ylabel='Local undercoverage error', title='ACI local errors and recency-weighted baseline')
    ax.legend()
    fig.tight_layout()
    fig.savefig(outdir / 'aci_local_error_vs_baseline.png', dpi=180)
    plt.close(fig)


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Device: {DEVICE}; dataset: {DATASET_PATH}')
    ds, scaler = load_dataset(DATASET_PATH)
    print(f'{len(ds)} sequence/target pairs; {len(ds)-TINIT} online predictions')
    traj = run_aci(ds, scaler)
    audit, summary = validate_eq23(traj)
    traj.to_csv(OUTPUT_DIR / 'aci_trajectory.csv', index=False)
    audit.to_csv(OUTPUT_DIR / 'eq23_by_update.csv', index=False)
    pd.DataFrame([summary]).to_csv(OUTPUT_DIR / 'eq23_summary.csv', index=False)
    make_plots(audit, OUTPUT_DIR)
    print('\n=== EQ (23): FINAL UPDATE i ===')
    print(f'Number of historical ACI failures |F_i| = {summary["final_F_i_size"]}')
    print(f'Number with LE_s > LEbar_i       = {summary["final_count_condition_true"]}')
    print(f'Fraction                         = {summary["final_fraction"]:.2%}')
    print(f'Minimum feasible p               = {summary["final_p_min"]:.4f}')
    print(f'Exists p < 2?                    = {summary["final_supports_p_lt_2"]}')
    print(f'Fraction of evaluated updates satisfying > 50% = {summary["fraction_updates_supporting_p_lt_2"]:.2%}')
    print(f'Eligible-only final fraction (exclude first {WINDOW-1} incomplete windows) = {summary["final_eligible_fraction"]:.2%}')
    print(f'Failure-time paired LE_i > LEbar_i = {summary["failure_time_fraction_LEi_gt_LEbar_i"]:.2%}')
    print(f'Files written to {OUTPUT_DIR.resolve()}')


if __name__ == '__main__':
    main()


Device: cuda; dataset: ../Datasets/50Hertz.csv
9480 sequence/target pairs; 4480 online predictions
ACI predictions: 200/4480 | alpha=0.1020
ACI predictions: 400/4480 | alpha=0.1120
ACI predictions: 600/4480 | alpha=0.1190
ACI predictions: 800/4480 | alpha=0.1210
ACI predictions: 1000/4480 | alpha=0.1280
ACI predictions: 1200/4480 | alpha=0.1430
ACI predictions: 1400/4480 | alpha=0.1510
ACI predictions: 1600/4480 | alpha=0.1480
ACI predictions: 1800/4480 | alpha=0.1520
ACI predictions: 2000/4480 | alpha=0.1580
ACI predictions: 2200/4480 | alpha=0.1690
ACI predictions: 2400/4480 | alpha=0.1590
ACI predictions: 2600/4480 | alpha=0.1420
ACI predictions: 2800/4480 | alpha=0.1320
ACI predictions: 3000/4480 | alpha=0.1380
ACI predictions: 3200/4480 | alpha=0.1330
ACI predictions: 3400/4480 | alpha=0.1470
ACI predictions: 3600/4480 | alpha=0.1490
ACI predictions: 3800/4480 | alpha=0.1290
ACI predictions: 4000/4480 | alpha=0.1110
ACI predictions: 4200/4480 | alpha=0.1110
ACI predictions: 4400/4